In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.filters.hp_filter import hpfilter

In [ ]:
def ensure_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        for c in ("DATE", "date", "Date"):
            if c in out.columns:
                out[c] = pd.to_datetime(out[c], errors="coerce")
                out = out.set_index(c)
                break
        else:
            out.index = pd.to_datetime(out.index, errors="coerce")
    return out.sort_index()

def auto_hp_lambda(idx: pd.DatetimeIndex, default_ppy: int= 12) -> float:
    """Lambda skaliert mit (periods per year / 4)^4, Basis: 1600"""
    freq = pd.infer_freq(idx)
    if freq is None:
        ppy = default_ppy
    else: 
        f = freq.upper()
        if "Q" in f: ppy = 4
        elif f in ("A","YE"): ppy = 1
        elif f.startswith("M"): ppy= 12
        elif f.startswith("W"): ppy=52
        elif f.startswith("D"): ppy = 365
        else: ppy = default_ppy
    return 1600.0 * (ppy / 4.0) ** 4

In [ ]:
df = pd.read_csv("data/aneta.csv")
df = ensure_datetime_index(df)
df = df[["CPI Index"]].copy()
df["CPI Index"] = df["CPI Index"].astype(float)
df.head()

In [ ]:
out = df.copy()
out["cpi_dlog"] = np.log(out["CPI Index"]).diff()
out["cpi_dlog_pp"] = 100 * out["cpi_dlog"]
out.tail(2)


In [ ]:
series = out["cpi_dlog_pp"].dropna()
lam = auto_hp_lambda(series.index)

cycle, trend = hpfilter(series, lamb=lam)
out.loc[series.index, "cpi_hp_trend"] = trend
out.loc[series.index, "cpi_hp_cycle"] = cycle
float(lam), out[["cpi_dlog_pp", "cpi_hp_trend", "cpi_hp_cycle"]].tail(3)

In [ ]:
out["cpi_hp_binary"] = (out["cpi_hp_cycle"] > 0.0).astype(int)
out[["cpi_dlog_pp", "cpi_hp_cycle", "cpi_hp_binary"]].dropna().head()

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.filters.hp_filter import hpfilter
import numpy as np

# --- Serie vorbereiten (falls du schon 'out' aus den vorherigen Schritten hast) ---
series = out["cpi_dlog_pp"].dropna()

# HP-Filter (für monatliche Daten: λ = 129600)
cycle, trend = hpfilter(series, lamb=129_600)

# --- Plot ---
plt.figure(figsize=(12,6))
plt.plot(series.index, series, label="d log CPI (pp)", color="tab:blue")
plt.plot(trend.index, trend, label="HP-Trend", color="tab:orange", linewidth=2)
plt.title("HP-Filter auf d log CPI - Monthly")
plt.xlabel("Zeit")
plt.ylabel("Prozentpunkte")
plt.legend()
plt.tight_layout()
plt.show()

# Optional: auch Zyklus separat anzeigen
plt.figure(figsize=(12,4))
plt.plot(cycle.index, cycle, label="HP-Cycle", color="tab:green")
plt.axhline(0, color="black", linewidth=1)
plt.title("HP-Cycle von d log CPI - Monthly")
plt.xlabel("Zeit")
plt.ylabel("Abweichung")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
out["cpi_hp_binary"].value_counts()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.filters.hp_filter import hpfilter
import warnings
from scipy.sparse import SparseEfficiencyWarning
warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)

df = ensure_datetime_index(df)
df = df[["CPI Index"]].astype(float).copy()

for freq, label in [("QE", "Quartal"), ("YE", "Jahr")]:
    # Aggregation (du nutzt mean; last wäre bei Indizes auch üblich)
    agg = df.resample(freq).mean()

    # d log (auf aggregierter Serie)
    series = np.log(agg["CPI Index"]).diff().dropna()

    # --- λ robust setzen: für Q/J feste Literaturlam, sonst auto als Fallback
    if freq == "QE":
        lam = 1600.0
    elif freq in ("A", "YE"):
        lam = 100.0
    else:
        lam = auto_hp_lambda(series.index)   # Fallback (z.B. für Monatsdaten)

    # HP-Filter
    cycle, trend = hpfilter(series, lamb=lam)

    # Plot: Original + Trend
    plt.figure(figsize=(12,5))
    plt.plot(series.index, series, label=f"d log CPI ({label})")
    plt.plot(trend.index, trend, label=f"HP-Trend λ={lam:.0f}", linewidth=2)
    plt.title(f"HP-Filter auf d log CPI – {label}")
    plt.xlabel("Zeit"); plt.legend(); plt.tight_layout(); plt.show()

    # Plot: Zyklus
    plt.figure(figsize=(12,3))
    plt.plot(cycle.index, cycle, label="HP-Zyklus")
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"HP-Zyklus von d log CPI – {label}")
    plt.xlabel("Zeit"); plt.legend(); plt.tight_layout(); plt.show()

    print(f"{label}: n={len(series)}, λ={lam:.0f}")

    binary = np.where(series >= trend, 1, -1)
    binary_series = series.copy()
    binary_series[:] = binary

    binary_series_name = f"bin_{label}"
    print(binary_series.head())

In [ ]:
data = pd.read_csv("data/aneta.csv")

In [ ]:
binary_series

In [ ]:
year_map = pd.Series(binary_series.values, index=binary_series.index.to_period("Y"))
df["CPI_bin_Jahr"] = df.index.to_period("Y").map(year_map).astype('float').astype('Int64')

In [ ]:
df

In [ ]:
data = data.merge(df, how="left")

In [ ]:
data = data.bfill()

In [ ]:
data.to_csv("data/mid.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.filters.cf_filter import cffilter  # Christiano–Fitzgerald

# Annahmen:
# - df existiert bereits und enthält "CPI Index"
# - ensure_datetime_index(df) ist bei dir schon definiert

# 1) Daten sauber
df = ensure_datetime_index(df)
df = df[["CPI Index"]].astype(float).copy()

# 2) Band-Pass Fenster (Business-Cycle-typisch: 1.5–8 Jahre)
#    Umgerechnet in Perioden:
#    - Monat:   18–96 (Monate)
#    - Quartal:  6–32 (Quartale)
#    - Jahr:     2–8  (Jahre)
bp_windows = {
    "M": (18, 96),
    "Q": (6, 32),
    "A": (2, 8),
}

# 3) Aggregationsstrategie für Q/J (bei Indizes oft 'last', du kannst 'mean' wählen)
agg_method = "mean"  # "last" oder "mean"

def resample_series(data, rule, how="mean"):
    if rule == "M":
        return data.copy()
    if how == "last":
        return data.resample(rule).last()
    elif how == "mean":
        return data.resample(rule).mean()
    else:
        raise ValueError("agg_method unbekannt: nutze 'last' oder 'mean'.")

# 4) Schleife über Frequenzen: M/Q/A
for rule, label in [("M", "Monat"), ("Q", "Quartal"), ("A", "Jahr")]:
    # Aggregation (nur für Q/A wirksam)
    agg = resample_series(df, rule, how=agg_method)

    # d log der aggregierten Serie
    series = np.log(agg["CPI Index"]).diff().dropna()
    low, high = bp_windows[rule]

    # Christiano–Fitzgerald Band-Pass (zwei-seitig; behält Randpunkte)
    # Rückgabe: cycle, trend
    cycle, trend = cffilter(series, low=low, high=high, drift=False)

    # --- Plot: Original + Trend ---
    plt.figure(figsize=(12,5))
    plt.plot(series.index, series, label=f"d log CPI ({label})")
    plt.plot(trend.index, trend, label=f"CF-Trend [{low}-{high} {rule}]", linewidth=2)
    plt.title(f"Christiano–Fitzgerald Band-Pass – {label}")
    plt.xlabel("Zeit"); plt.legend(); plt.tight_layout(); plt.show()

    # --- Plot: Zyklus ---
    plt.figure(figsize=(12,3))
    plt.plot(cycle.index, cycle, label="CF-Zyklus")
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"CF-Zyklus von d log CPI – {label}")
    plt.xlabel("Zeit"); plt.legend(); plt.tight_layout(); plt.show()

    # Optional: kurze Statusausgabe
    print(f"{label}: n={len(series)}, Bandpass [{low}, {high}] in {rule}-Perioden")

# ---------------------------------------------------------
# OPTIONAL: Binärvariable (Zyklus>0) für jede Frequenz:
#    hp_like_binary = (cycle > 0).astype(int)
# --------------------------------------------------------

In [ ]:
data = pd.read_csv("data/final7.csv")

In [ ]:
data["pct"] = data["XAU Curncy"].pct_change().dropna()*100

In [ ]:
data["pct"].mean()

In [ ]:
# ============================
# Baxter–King Band-Pass Filter
# ============================
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.filters.bk_filter import bkfilter

# Annahme: df existiert, enthält "CPI Index"
df = ensure_datetime_index(df)
df = df[["CPI Index"]].astype(float).copy()

def resample_series(data, rule, how="mean"):
    if rule == "M":
        return data.copy()
    if how == "last":
        return data.resample(rule).last()
    elif how == "mean":
        return data.resample(rule).mean()
    else:
        raise ValueError("agg_method unbekannt: 'last' oder 'mean'.")

# Passband: 1.5–8 Jahre (in Perioden)
bk_windows = {
    "M": (18, 96),   # Monate
    "Q": (6, 32),    # Quartale
    "A": (2, 8),     # Jahre
}

# K = Anzahl Leads/Lags für Filter (mehr = glatter, aber mehr Randverlust)
bk_K = {"M": 24, "Q": 8, "A": 2}

# Aggregationsregel für Q/A (bei Indizes oft 'last')
agg_method = "mean"

for rule, label in [("M", "Monat"), ("Q", "Quartal"), ("A", "Jahr")]:
    # Resample
    agg = resample_series(df, rule, how=agg_method)
    # d log CPI
    series = np.log(agg["CPI Index"]).diff().dropna()

    low, high = bk_windows[rule]
    K = bk_K[rule]

    # BK-Filter: liefert Zyklus, aber kürzt Ränder (K Beobachtungen weg)
    cycle = bkfilter(series, low=low, high=high, K=K)
    trend = series.loc[cycle.index] - cycle   # Trend = Original - Zyklus

    # ---- Plot: Original + Trend ----
    plt.figure(figsize=(12,5))
    plt.plot(series.index, series, label=f"d log CPI ({label})")
    plt.plot(trend.index, trend, label=f"BK-Trend [{low}-{high} {rule}], K={K}", linewidth=2)
    plt.title(f"Baxter–King Filter – {label}")
    plt.xlabel("Zeit"); plt.legend(); plt.tight_layout(); plt.show()

    # ---- Plot: Zyklus ----
    plt.figure(figsize=(12,3))
    plt.plot(cycle.index, cycle, label="BK-Zyklus")
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"BK-Zyklus von d log CPI – {label}")
    plt.xlabel("Zeit"); plt.legend(); plt.tight_layout(); plt.show()

    print(f"{label}: BK-Fenster [{low}, {high}], K={K}, gültige Punkte={len(cycle)}")

In [ ]:
df = pd.read_csv("data/aneta.csv")